In [ ]:
!pip install -q streamlit

In [ ]:
!mkdir -p data models reports/figures

In [ ]:
%%writefile text_utils.py
"""
Shared text-preprocessing helpers for the sentiment analysis project.

This lives in its own module (not in the notebook) so that the pickled
scikit-learn pipeline can find `clean_text` again when it is loaded by
app.py or predict.py.
"""
import re

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Negations flip the meaning of a sentence ("good" vs "not good"), so we must
# NOT throw them away together with the other stop-words.
NEGATIONS = {"no", "not", "nor", "never", "none", "nothing", "nobody",
             "neither", "cannot", "without"}

# Standard English stop-words minus the negations.
STOP_WORDS = frozenset(ENGLISH_STOP_WORDS) - NEGATIONS

_CONTRACTIONS = [
    (r"\bwon't\b", "will not"),
    (r"\bcan't\b", "can not"),
    (r"\bain't\b", "is not"),
    (r"n't\b", " not"),          # didn't -> did not, isn't -> is not ...
    (r"'(ve|ll|re|d|s|m)\b", ""),  # I've -> I, it's -> it (stop-words anyway)
]
_NON_LETTERS = re.compile(r"[^a-z\s]")
_SPACES = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """Lower-case, expand n't, strip punctuation/digits, drop stop-words.

    Example
    -------
    >>> clean_text("The battery DIDN'T last 2 days!")
    'battery did not days'
    """
    text = str(text).lower().replace("’", "'")
    for pattern, repl in _CONTRACTIONS:
        text = re.sub(pattern, repl, text)
    text = _NON_LETTERS.sub(" ", text)
    tokens = [t for t in _SPACES.split(text) if len(t) > 1 and t not in STOP_WORDS]
    return " ".join(tokens)

In [ ]:
%%writefile data_utils.py
"""
Data loading and leakage-aware splitting helpers.

Why this module exists
----------------------
The provided dataset has 10,000 rows but only 300 *distinct* review texts
(100 per class), and those texts are built from just 72 distinct sentences.
A normal random train/test split therefore puts the very same sentences in
both train and test and reports ~100 % accuracy that says nothing about
real-world performance.

To measure generalisation honestly we split by *sentence*: every sentence in
the held-out set is guaranteed to be absent from the training data.
"""
import re
import random
import pandas as pd

LABELS = ["Negative", "Neutral", "Positive"]
_SENT = re.compile(r"[^.!?]+[.!?]")


def load_data(path="data/reviews.csv") -> pd.DataFrame:
    return pd.read_csv(path)


def split_sentences(text: str):
    return [s.strip() for s in _SENT.findall(text)]


def add_sentence_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["sentences"] = df["review_text"].apply(split_sentences)
    return df


def _sentences_per_class(df):
    return {lab: sorted({s for ss in g["sentences"] for s in ss})
            for lab, g in df.groupby("sentiment")}


def _split_by_held_sentences(df, held):
    has_held = df["sentences"].apply(lambda ss: any(s in held for s in ss))
    return (~has_held).to_numpy(), has_held.to_numpy()


def sentence_holdout_split(df, held_per_class=6, seed=42):
    """Final train/test split. Test = reviews that contain any held-out sentence;
    train = reviews that contain none of them. Returns two boolean masks."""
    rng = random.Random(seed)
    held = set()
    for lab, sents in _sentences_per_class(df).items():
        held |= set(rng.sample(sents, held_per_class))
    return _split_by_held_sentences(df, held)


def sentence_grouped_folds(df, n_folds=3, seed=42):
    """Cross-validation folds with no sentence shared between train and
    validation part of a fold. Returns a list of (train_idx, val_idx) that can
    be passed straight to GridSearchCV / cross_val_score as `cv=`.
    `df` must have a clean 0..n-1 index (use reset_index(drop=True))."""
    rng = random.Random(seed)
    per_class = _sentences_per_class(df)
    fold_sets = [set() for _ in range(n_folds)]
    for lab, sents in per_class.items():
        sents = sents[:]
        rng.shuffle(sents)
        for i, s in enumerate(sents):
            fold_sets[i % n_folds].add(s)
    folds = []
    for held in fold_sets:
        tr, va = _split_by_held_sentences(df, held)
        folds.append((tr.nonzero()[0], va.nonzero()[0]))
    return folds

In [ ]:
%%writefile predict.py
"""
Inference helpers shared by the Streamlit app, the notebook and the CLI.

    python predict.py "The battery died after two days, very disappointed"
"""
import json
import sys
from pathlib import Path

import joblib
import numpy as np

from text_utils import clean_text  # noqa: F401  (needed to unpickle the pipeline)

ROOT = Path(__file__).resolve().parent
MODEL_PATH = ROOT / "models" / "sentiment_pipeline.pkl"
META_PATH = ROOT / "models" / "model_metadata.json"


def load_pipeline(path=MODEL_PATH):
    """Load the trained scikit-learn pipeline (vectoriser + classifier)."""
    if not Path(path).exists():
        raise FileNotFoundError(
            f"{path} not found. Run notebooks/sentiment_analysis.ipynb or `python train.py` first.")
    return joblib.load(path)


def load_metadata(path=META_PATH):
    p = Path(path)
    return json.loads(p.read_text()) if p.exists() else {}


def explain(pipeline, text, label, top_n=5):
    """Words/phrases that pushed the prediction towards `label`
    (coefficient x TF-IDF weight). Works for linear models only."""
    clf = pipeline.named_steps["clf"]
    if not hasattr(clf, "coef_"):
        return []
    vec = pipeline.named_steps["vec"]
    x = vec.transform([text])
    names = vec.get_feature_names_out()
    row = list(clf.classes_).index(label)
    contrib = x.multiply(clf.coef_[row]).tocoo()
    pairs = sorted(zip(contrib.col, contrib.data), key=lambda t: -t[1])
    return [(names[j], float(v)) for j, v in pairs if v > 0][:top_n]


def predict_sentiment(text, pipeline=None):
    """Return label, confidence, per-class probabilities and key terms."""
    pipeline = pipeline or load_pipeline()
    probs = pipeline.predict_proba([text])[0]
    classes = list(pipeline.classes_)
    best = int(np.argmax(probs))
    label = classes[best]
    return {
        "label": label,
        "confidence": float(probs[best]),
        "probabilities": {c: float(p) for c, p in zip(classes, probs)},
        "cleaned_text": clean_text(text),
        "key_terms": explain(pipeline, text, label),
    }


if __name__ == "__main__":
    review = " ".join(sys.argv[1:]) or "The product works well and I am happy with it."
    result = predict_sentiment(review)
    print(f"Review     : {review}")
    print(f"Sentiment  : {result['label']}  ({result['confidence']:.1%} confidence)")
    for c, p in result["probabilities"].items():
        print(f"  {c:<9}{p:6.1%}")

In [ ]:
%%writefile data/external_test_reviews.csv
review_text,sentiment
"I absolutely love this, it works great and looks fantastic.",Positive
"Really happy with my purchase, would definitely buy again.",Positive
The quality is outstanding and it arrived quickly.,Positive
"Best purchase I've made this year, highly recommended.",Positive
"Super comfortable and well made, I use it every day.",Positive
Great value for money and the customer service was friendly.,Positive
It does exactly what it promises and the performance is excellent.,Positive
"Very pleased, the product feels sturdy and premium.",Positive
Fantastic experience from ordering to delivery.,Positive
I was skeptical at first but it turned out to be wonderful.,Positive
"The app is smooth, fast and easy to use.",Positive
"Beautiful design and it works perfectly, five stars.",Positive
"Delivery was fast and everything was packed carefully, very satisfied.",Positive
This exceeded what I expected for the price.,Positive
The battery lasts all day and the screen is gorgeous.,Positive
"Impressive quality, I'm glad I chose this one.",Positive
"Everything about it is good, no complaints at all.",Positive
My family loves it and we would recommend it to friends.,Positive
Setup was effortless and it has been reliable ever since.,Positive
"Such a great find, it is worth every penny.",Positive
The staff were helpful and the whole process was pleasant.,Positive
"Delighted with the results, it is better than I hoped.",Positive
"Soft fabric, perfect fit and lovely colour.",Positive
Works like a charm and feels really durable.,Positive
"Wonderful product, I could not be happier.",Positive
The taste is delicious and the packaging is neat.,Positive
"Smooth checkout and a great product, I am impressed.",Positive
Top notch build and very good sound quality.,Positive
"I'm really enjoying it, it makes my life easier.",Positive
"Excellent service, they solved my issue immediately.",Positive
"This is terrible, it broke after two days.",Negative
"Very disappointed, the quality is awful and not worth the money.",Negative
"I would not recommend this to anyone, total waste of money.",Negative
The product arrived damaged and customer service ignored me.,Negative
"Worst purchase ever, it does not work at all.",Negative
"It stopped charging within a week, really frustrating.",Negative
Delivery took forever and the package was crushed.,Negative
The material feels cheap and the stitching is falling apart.,Negative
"Poor design, uncomfortable and difficult to use.",Negative
The app keeps crashing and support never replied.,Negative
"I regret buying this, it is nothing like the pictures.",Negative
"Awful experience, rude staff and a long wait.",Negative
"Not good at all, the battery dies in an hour.",Negative
"The food was cold and tasted bad, never ordering again.",Negative
"It is noisy, slow and constantly freezes.",Negative
"Complete rip-off, it does not do what it claims.",Negative
The size was wrong and returning it was a nightmare.,Negative
"Disgusting smell and poor packaging, very unhappy.",Negative
"Useless product, I want a refund.",Negative
"The screen cracked on the first day, terrible quality.",Negative
"Overpriced and underwhelming, I would not buy it again.",Negative
The instructions were confusing and the parts did not fit.,Negative
"Horrible customer support, they kept me waiting for hours.",Negative
"It looked nice but it fell apart quickly, a big disappointment.",Negative
"I am extremely unhappy, nothing works as described.",Negative
"Bad quality and late delivery, a really poor experience.",Negative
The website is slow and the order was cancelled without notice.,Negative
"Not worth it, it feels flimsy and breaks easily.",Negative
They sent the wrong item and refused to fix it.,Negative
"Frustrating and unreliable, I cannot recommend it.",Negative
"The package contains a charger, a cable and a user manual.",Neutral
I ordered it on Tuesday and it came on Friday.,Neutral
"The jacket is available in blue, black and grey.",Neutral
It has a 6.1 inch display and 128 GB of storage.,Neutral
I have only had it for a day so I cannot say much yet.,Neutral
The store opens at nine and closes at six.,Neutral
This is the second item I have bought from this seller.,Neutral
The book has 320 pages and a paperback cover.,Neutral
It comes in a box with standard packaging.,Neutral
The app allows users to track their orders.,Neutral
I am still testing it and will update my review later.,Neutral
The table is made of wood and measures 120 centimetres.,Neutral
The shipment arrived in two separate parcels.,Neutral
It works with both Android and iOS devices.,Neutral
The product description lists the weight as 500 grams.,Neutral
I purchased it for my brother as a gift.,Neutral
The hotel is located near the central station.,Neutral
There are three size options and two colour options.,Neutral
The warranty period is twelve months.,Neutral
It requires two AA batteries which are not included.,Neutral
I used it a few times and it is about what I expected.,Neutral
The order confirmation was sent by email.,Neutral
The service has a monthly and an annual plan.,Neutral
It is neither better nor worse than my previous one.,Neutral
The cable is one metre long and has a USB connector.,Neutral
I picked it up from the store this afternoon.,Neutral
The set includes four plates and four bowls.,Neutral
Delivery options include standard and express shipping.,Neutral
"It is an average product, nothing special either way.",Neutral
The device has a power button on the left side.,Neutral

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
shutil.move(next(iter(uploaded)), "data/reviews.csv")

In [ ]:
import sys, json, warnings, time, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from IPython.display import display

warnings.filterwarnings("ignore")

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))               # so we can import our own helper modules

from text_utils import clean_text, STOP_WORDS
from data_utils import (load_data, add_sentence_column, sentence_holdout_split,
                        sentence_grouped_folds, LABELS)

SEED = 42
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
COLORS = {"Negative": "#d1495b", "Neutral": "#8d99ae", "Positive": "#2a9d8f"}

def savefig(name):
    plt.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")

METRICS = {}          # everything we want to reuse in the evaluation report
print("Setup done. Project root:", ROOT)

In [ ]:
df = load_data(ROOT / "data" / "reviews.csv")
print("Shape:", df.shape)
df.head(8)

In [ ]:
df.info()

In [ ]:
print("Missing values per column")
print(df.isna().sum().to_string())

print("\nFully duplicated rows      :", df.duplicated().sum())
print("Duplicated review_id       :", df["review_id"].duplicated().sum())
print("Duplicated review_text     :", df["review_text"].duplicated().sum())
print("Distinct review texts      :", df["review_text"].nunique())

ws = (df["review_text"] != df["review_text"].str.strip()).sum()
print("\nTexts with leading/trailing spaces:", ws)
print("Distinct sentiment labels  :", sorted(df["sentiment"].unique()))

In [ ]:
df["review_text"] = df["review_text"].str.strip()                 # harmless safety clean
df["sentiment"] = df["sentiment"].str.strip().str.title()          # consistent label spelling
assert set(df["sentiment"]) == set(LABELS)

df = add_sentence_column(df)        # split every review into its sentences

per_class = (df.explode("sentences")
               .groupby("sentiment")
               .agg(distinct_texts=("review_text", "nunique"),
                    distinct_sentences=("sentences", "nunique")))
display(per_class)

sent_labels = df.explode("sentences").groupby("sentences")["sentiment"].nunique()
print("Distinct sentences overall:", len(sent_labels))
print("Sentences that appear under more than one label:", int((sent_labels > 1).sum()))
print("Sentences per review:", df["sentences"].str.len().value_counts().sort_index().to_dict())

In [ ]:
counts = df["sentiment"].value_counts().reindex(LABELS)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=[COLORS[l] for l in counts.index])
ax.bar_label(bars, fmt="{:,.0f}", padding=3)
ax.set_title("Reviews per sentiment class")
ax.set_ylabel("Number of reviews")
ax.set_ylim(0, counts.max() * 1.12)
savefig("01_class_distribution.png")
plt.show()
print((counts / len(df) * 100).round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, col in zip(axes, ["category", "source"]):
    ct = pd.crosstab(df[col], df["sentiment"], normalize="index")[LABELS] * 100
    ct.plot(kind="barh", stacked=True, ax=ax, color=[COLORS[l] for l in LABELS], width=0.75)
    ax.set_title(f"Sentiment mix by {col}")
    ax.set_xlabel("% of reviews"); ax.set_ylabel("")
    ax.legend(title="", loc="lower right", fontsize=8)
plt.tight_layout()
savefig("02_metadata_vs_sentiment.png")
plt.show()

for col in ["category", "source"]:
    chi2, p, dof, _ = chi2_contingency(pd.crosstab(df[col], df["sentiment"]))
    print(f"Chi-square test  {col:<9} vs sentiment:  chi2={chi2:6.2f}, dof={dof}, p-value={p:.3f}")

In [ ]:
df["n_words"] = df["review_text"].str.split().str.len()
df["n_chars"] = df["review_text"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sns.boxplot(data=df, x="sentiment", y="n_words", order=LABELS, palette=COLORS, ax=axes[0])
axes[0].set_title("Words per review"); axes[0].set_xlabel("")
for lab in LABELS:
    axes[1].hist(df.loc[df.sentiment == lab, "n_words"], bins=range(5, 28), alpha=.55,
                 color=COLORS[lab], label=lab)
axes[1].set_title("Distribution of review length"); axes[1].set_xlabel("words"); axes[1].legend()
plt.tight_layout(); savefig("03_review_length.png"); plt.show()

display(df.groupby("sentiment")[["n_words", "n_chars"]].agg(["mean", "median", "min", "max"]).round(1))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharex=False)
top_words = {}
for ax, lab in zip(axes, LABELS):
    cv = CountVectorizer(preprocessor=clean_text)
    # count each DISTINCT text once so the 33x repetition does not distort the ranking
    texts = df.loc[df.sentiment == lab, "review_text"].drop_duplicates()
    m = cv.fit_transform(texts)
    freq = pd.Series(np.asarray(m.sum(axis=0)).ravel(), index=cv.get_feature_names_out()).nlargest(10)
    top_words[lab] = freq
    ax.barh(freq.index[::-1], freq.values[::-1], color=COLORS[lab])
    ax.set_title(f"{lab}: top words")
plt.tight_layout(); savefig("04_top_words.png"); plt.show()

In [ ]:
examples = [
    "The battery DIDN'T last 2 days!!",
    "I've used it once so far. I have not used it enough to form a strong opinion.",
    "Excellent quality for the price, and I would happily buy it again.",
]
for e in examples:
    print("RAW    :", e)
    print("CLEANED:", clean_text(e))
    print()

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def sklearn_default_cleanup(t):
    return " ".join(w for w in t.lower().replace(".", "").split() if w not in ENGLISH_STOP_WORDS)

for t in ["The product is good.", "The product is not good."]:
    print(f"{t!r:35} default stop-words -> {sklearn_default_cleanup(t)!r:22} ours -> {clean_text(t)!r}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

Xtr, Xte, ytr, yte = train_test_split(df["review_text"], df["sentiment"], test_size=0.2,
                                      stratify=df["sentiment"], random_state=SEED)

overlap = len(set(Xte) & set(Xtr)) / Xte.nunique()
naive = Pipeline([("vec", TfidfVectorizer(preprocessor=clean_text)),
                  ("clf", LogisticRegression(max_iter=3000))]).fit(Xtr, ytr)
naive_acc = naive.score(Xte, yte)

print(f"Distinct test texts that ALSO appear in the training set: {overlap:.0%}")
print(f"Accuracy of a plain random split                         : {naive_acc:.1%}")
METRICS["naive_random_split_accuracy"] = naive_acc
METRICS["naive_split_text_overlap"] = overlap

In [ ]:
train_mask, test_mask = sentence_holdout_split(df, held_per_class=6, seed=SEED)
train_df = df[train_mask].reset_index(drop=True)
test_df = df[test_mask].reset_index(drop=True)

train_sents = {s for ss in train_df["sentences"] for s in ss}
test_sents = {s for ss in test_df["sentences"] for s in ss}
unseen = test_sents - train_sents
assert all(any(s in unseen for s in ss) for ss in test_df["sentences"]), "every test review must contain an unseen sentence"

print(f"Train reviews: {len(train_df):,}   Test reviews: {len(test_df):,}")
print(f"Distinct sentences  train: {len(train_sents)}   test: {len(test_sents)}   never seen in training: {len(unseen)}")
print("Share of test reviews that are exact copies of a training review:",
      f"{test_df['review_text'].isin(set(train_df['review_text'])).mean():.0%}")
display(pd.concat({"train": train_df["sentiment"].value_counts(),
                   "test": test_df["sentiment"].value_counts()}, axis=1).loc[LABELS])

# leak-free CV folds on the training part (3 folds, no sentence shared between fit and validation part)
folds = sentence_grouped_folds(train_df, n_folds=3, seed=SEED)
print("\nCV folds (train size / validation size):", [(len(a), len(b)) for a, b in folds])
METRICS["n_train"], METRICS["n_test"] = len(train_df), len(test_df)

In [ ]:
from sklearn.model_selection import cross_val_score

ext = pd.read_csv(ROOT / "data" / "external_test_reviews.csv")     # 90 hand-written reviews (used in Section 10)

variants = {
    "A) raw text, lower-case only (no cleaning)":      TfidfVectorizer(),
    "B) scikit-learn stop-words (drops 'not')":        TfidfVectorizer(stop_words="english"),
    "C) my clean_text (keeps negations)":              TfidfVectorizer(preprocessor=clean_text),
}
rows = []
for name, vec in variants.items():
    pipe = Pipeline([("vec", vec), ("clf", LogisticRegression(max_iter=3000))])
    s = cross_val_score(pipe, train_df["review_text"], train_df["sentiment"], cv=folds, scoring="f1_macro")
    pipe.fit(train_df["review_text"], train_df["sentiment"])
    rows.append({"preprocessing": name, "CV macro-F1": s.mean(), "CV std": s.std(),
                 "hand-written accuracy (info only)": pipe.score(ext["review_text"], ext["sentiment"])})
ablation = pd.DataFrame(rows).set_index("preprocessing").round(3)
METRICS["preprocessing_ablation"] = ablation.reset_index().to_dict("records")
display(ablation)

In [ ]:
demo = ["good product good price", "not good bad price", "average product"]
bow = CountVectorizer(preprocessor=clean_text)
tf = TfidfVectorizer(preprocessor=clean_text)
print("Documents:", demo, "\n")
print("Bag of Words (counts)")
display(pd.DataFrame(bow.fit_transform(demo).toarray(), columns=bow.get_feature_names_out(), index=[f"doc{i}" for i in range(3)]))
print("TF-IDF (weights)")
display(pd.DataFrame(tf.fit_transform(demo).toarray().round(2), columns=tf.get_feature_names_out(), index=[f"doc{i}" for i in range(3)]))

In [ ]:
for rng in [(1, 1), (1, 2)]:
    v = TfidfVectorizer(preprocessor=clean_text, ngram_range=rng)
    X = v.fit_transform(train_df["review_text"])
    sparsity = 1 - X.nnz / (X.shape[0] * X.shape[1])
    print(f"ngram_range={rng}:  vocabulary={X.shape[1]:>4} features,  {sparsity:.1%} of the matrix is zeros")

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score)
from sklearn.preprocessing import label_binarize

def make_vectorizer(kind, **kw):
    cls = CountVectorizer if kind == "bow" else TfidfVectorizer
    return cls(preprocessor=clean_text, **kw)

def make_pipeline(kind, clf, **vec_kw):
    return Pipeline([("vec", make_vectorizer(kind, **vec_kw)), ("clf", clf)])

MODELS = {
    "Naive Bayes":         lambda: MultinomialNB(),
    "Logistic Regression": lambda: LogisticRegression(max_iter=3000),
    "Linear SVM":          lambda: LinearSVC(),
    "Random Forest":       lambda: RandomForestClassifier(n_estimators=200, random_state=SEED),
}
X_train, y_train = train_df["review_text"], train_df["sentiment"]
X_test, y_test = test_df["review_text"], test_df["sentiment"]

def test_scores(pipe):
    pred = pipe.predict(X_test)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    return dict(test_accuracy=accuracy_score(y_test, pred), test_precision=p, test_recall=r, test_f1=f)

rows = []
for kind, kind_name in [("bow", "Bag of Words"), ("tfidf", "TF-IDF")]:
    for name, mk in MODELS.items():
        pipe = make_pipeline(kind, mk())
        t0 = time.time()
        cv = cross_validate(pipe, X_train, y_train, cv=folds, scoring={"f1": "f1_macro", "acc": "accuracy"}, n_jobs=-1)
        pipe.fit(X_train, y_train)
        rows.append({"features": kind_name, "model": name,
                     "cv_f1": cv["test_f1"].mean(), "cv_f1_std": cv["test_f1"].std(),
                     "cv_accuracy": cv["test_acc"].mean(),
                     **test_scores(pipe), "seconds": time.time() - t0})

baseline = pd.DataFrame(rows)
baseline.to_csv(ROOT / "reports" / "baseline_model_comparison.csv", index=False)
METRICS["baseline"] = baseline.round(4).to_dict("records")
show = baseline.drop(columns="seconds").round(3)
display(show.style.background_gradient(subset=["cv_f1", "test_f1"], cmap="Greens").format(precision=3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
piv = baseline.pivot(index="model", columns="features", values="cv_f1").loc[list(MODELS)]
err = baseline.pivot(index="model", columns="features", values="cv_f1_std").loc[list(MODELS)]
piv.plot(kind="bar", yerr=err, ax=ax, capsize=3, color=["#6c8ebf", "#e29578"], width=0.7)
ax.set_ylim(0.6, 1.0); ax.set_ylabel("CV macro-F1"); ax.set_xlabel("")
ax.set_title("Baseline models: Bag of Words vs TF-IDF (leak-free 3-fold CV, error bars = std)")
plt.xticks(rotation=0); ax.legend(title="")
plt.tight_layout(); savefig("05_baseline_comparison.png"); plt.show()

In [ ]:
def get_scores(model, X):
    # probabilities if the model has them, otherwise the SVM decision scores
    return model.predict_proba(X) if hasattr(model, "predict_proba") else model.decision_function(X)

def evaluate(model, X, y, title, save_as=None):
    pred = model.predict(X)
    print(classification_report(y, pred, labels=LABELS, digits=3, zero_division=0))
    cm = confusion_matrix(y, pred, labels=LABELS)
    scores = get_scores(model, X)
    Y = label_binarize(y, classes=LABELS)

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=axes[0], cmap="Blues", colorbar=False, values_format="d")
    axes[0].set_title(f"Confusion matrix – {title}"); axes[0].grid(False)
    aucs = {}
    for i, lab in enumerate(LABELS):
        fpr, tpr, _ = roc_curve(Y[:, i], scores[:, i])
        aucs[lab] = auc(fpr, tpr)
        axes[1].plot(fpr, tpr, color=COLORS[lab], lw=2, label=f"{lab} (AUC = {aucs[lab]:.3f})")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
    axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
    axes[1].set_title(f"ROC curves (one-vs-rest) – {title}"); axes[1].legend(loc="lower right")
    plt.tight_layout()
    if save_as: savefig(save_as)
    plt.show()

    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, pred), "precision_macro": p, "recall_macro": r, "f1_macro": f,
            "roc_auc_macro": roc_auc_score(Y, scores, average="macro"), "auc_per_class": aucs, "confusion_matrix": cm.tolist()}

best_base = baseline.sort_values("cv_f1", ascending=False).iloc[0]
print(f"Best baseline by CV macro-F1: {best_base['model']} + {best_base['features']}  (CV F1 = {best_base['cv_f1']:.3f})\n")
kind = "bow" if best_base["features"] == "Bag of Words" else "tfidf"
best_base_pipe = make_pipeline(kind, MODELS[best_base["model"]]()).fit(X_train, y_train)
base_eval = evaluate(best_base_pipe, X_test, y_test, f"baseline {best_base['model']}", "06_baseline_eval.png")

In [ ]:
from sklearn.model_selection import StratifiedKFold

pipe = make_pipeline("tfidf", LogisticRegression(max_iter=3000))
std_cv = cross_val_score(pipe, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="f1_macro")
grp_cv = cross_val_score(pipe, X_train, y_train, cv=folds, scoring="f1_macro")
print(f"Ordinary stratified 5-fold CV : macro-F1 = {std_cv.mean():.3f}  (folds: {np.round(std_cv, 3)})")
print(f"Sentence-grouped 3-fold CV    : macro-F1 = {grp_cv.mean():.3f}  (folds: {np.round(grp_cv, 3)})")
METRICS["cv_ordinary"] = float(std_cv.mean()); METRICS["cv_grouped"] = float(grp_cv.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV

FAMILIES = {
    "Naive Bayes":         (MultinomialNB(),                         {"clf__alpha": [0.1, 0.5, 1.0, 2.0]}),
    "Logistic Regression": (LogisticRegression(max_iter=3000),       {"clf__C": [0.1, 1, 10, 100]}),
    "Linear SVM":          (LinearSVC(),                             {"clf__C": [0.01, 0.1, 1, 10]}),
    "Random Forest":       (RandomForestClassifier(random_state=SEED), {"clf__n_estimators": [100, 300],
                                                                         "clf__max_depth": [None, 20]}),
}
searches, rows = {}, []
for name, (clf, params) in FAMILIES.items():
    grid = {"vec": [make_vectorizer("bow"), make_vectorizer("tfidf")],
            "vec__ngram_range": [(1, 1), (1, 2)], **params}
    gs = GridSearchCV(Pipeline([("vec", make_vectorizer("tfidf")), ("clf", clf)]), grid, cv=folds,
                      scoring={"f1_macro": "f1_macro", "accuracy": "accuracy"}, refit="f1_macro", n_jobs=-1)
    gs.fit(X_train, y_train)
    searches[name] = gs
    bp = gs.best_params_
    i = gs.best_index_
    rows.append({"model": name,
                 "features": "TF-IDF" if isinstance(bp["vec"], TfidfVectorizer) else "Bag of Words",
                 "ngram_range": str(bp["vec__ngram_range"]),
                 "best_params": {k.replace("clf__", ""): v for k, v in bp.items() if k.startswith("clf__")},
                 "n_candidates": len(gs.cv_results_["params"]),
                 "cv_f1": gs.cv_results_["mean_test_f1_macro"][i], "cv_f1_std": gs.cv_results_["std_test_f1_macro"][i],
                 "cv_accuracy": gs.cv_results_["mean_test_accuracy"][i],
                 **test_scores(gs.best_estimator_)})

tuned = pd.DataFrame(rows).sort_values("cv_f1", ascending=False).reset_index(drop=True)
tuned.to_csv(ROOT / "reports" / "tuned_model_comparison.csv", index=False)
METRICS["tuned"] = json.loads(tuned.round(4).to_json(orient="records"))
display(tuned.round(3))

In [ ]:
top = tuned.iloc[0]
threshold = top["cv_f1"] - top["cv_f1_std"]
tied = tuned[tuned["cv_f1"] >= threshold]["model"].tolist()
PREFERENCE = ["Logistic Regression", "Linear SVM", "Naive Bayes", "Random Forest"]
final_name = next(m for m in PREFERENCE if m in tied)

print(f"Highest CV macro-F1: {top['model']} = {top['cv_f1']:.3f} (std {top['cv_f1_std']:.3f}); tie threshold = {threshold:.3f}")
print("Statistically tied models:", tied)
print("Selected final model     :", final_name)
final_search = searches[final_name]
print("Best parameters          :", {k: (type(v).__name__ if k == 'vec' else v) for k, v in final_search.best_params_.items()})
METRICS["final_model"] = final_name
METRICS["final_params"] = {k: (type(v).__name__ if k == "vec" else (list(v) if isinstance(v, tuple) else v))
                           for k, v in final_search.best_params_.items()}
METRICS["final_cv_f1"] = float(tuned.loc[tuned.model == final_name, "cv_f1"].iloc[0])
METRICS["final_cv_f1_std"] = float(tuned.loc[tuned.model == final_name, "cv_f1_std"].iloc[0])

In [ ]:
res = pd.DataFrame(final_search.cv_results_)
res["features"] = res["param_vec"].apply(lambda v: "TF-IDF" if isinstance(v, TfidfVectorizer) else "BoW")
res["ngram"] = res["param_vec__ngram_range"].astype(str)
res["setting"] = res["features"] + " " + res["ngram"]
hp = [c for c in res.columns if c.startswith("param_clf__")][0]
res["hp"] = res[hp]

fig, ax = plt.subplots(figsize=(8, 4.4))
for setting, g in res.groupby("setting"):
    g = g.sort_values("hp")
    ax.errorbar(g["hp"].astype(float), g["mean_test_f1_macro"], yerr=g["std_test_f1_macro"], marker="o", capsize=3, label=setting)
ax.set_xscale("log" if res["hp"].astype(float).min() > 0 else "linear")
ax.set_xlabel(hp.replace("param_clf__", "") + (" (regularisation strength, higher = weaker regularisation)" if "C" in hp else ""))
ax.set_ylabel("CV macro-F1"); ax.set_title(f"Grid search results – {final_name}"); ax.legend(fontsize=8)
plt.tight_layout(); savefig("07_grid_search_final_model.png"); plt.show()

In [ ]:
final_pipe = final_search.best_estimator_
final_eval = evaluate(final_pipe, X_test, y_test, f"tuned {final_name}", "08_final_eval.png")
METRICS["final_test"] = final_eval
print({k: round(v, 4) for k, v in final_eval.items() if isinstance(v, float)})

In [ ]:
cmp = pd.DataFrame({
    "baseline (default params, best vectoriser)": {k: base_eval[k] for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_macro"]},
    "tuned final model": {k: final_eval[k] for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_macro"]},
}).round(4)
METRICS["baseline_vs_tuned"] = cmp.to_dict()
display(cmp)

In [ ]:
test_df["pred"] = final_pipe.predict(X_test)
test_df["conf"] = final_pipe.predict_proba(X_test).max(axis=1)
errors = test_df[test_df["pred"] != test_df["sentiment"]]
print(f"Misclassified test reviews: {len(errors)} of {len(test_df)} ({len(errors)/len(test_df):.1%}),  "
      f"{errors['review_text'].nunique()} distinct texts")
print("\nWhich confusions happen most? (true -> predicted)")
display(errors.groupby(["sentiment", "pred"]).size().rename("count").reset_index().sort_values("count", ascending=False))

print("Most frequent mistakes (distinct texts):")
worst = (errors.groupby(["review_text", "sentiment", "pred"]).agg(times=("conf", "size"), confidence=("conf", "mean"))
               .reset_index().sort_values("times", ascending=False).head(8))
pd.set_option("display.max_colwidth", 110)
display(worst.round(2))

In [ ]:
vec, clf = final_pipe.named_steps["vec"], final_pipe.named_steps["clf"]
names = vec.get_feature_names_out()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, (i, lab) in zip(axes, enumerate(clf.classes_)):
    top = pd.Series(clf.coef_[i], index=names).nlargest(10)
    ax.barh(top.index[::-1], top.values[::-1], color=COLORS[lab])
    ax.set_title(f"Strongest terms for {lab}")
plt.tight_layout(); savefig("09_top_coefficients.png"); plt.show()

In [ ]:
ext_pred = final_pipe.predict(ext["review_text"])
print(classification_report(ext["sentiment"], ext_pred, labels=LABELS, digits=3, zero_division=0))

cm = confusion_matrix(ext["sentiment"], ext_pred, labels=LABELS)
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Oranges", colorbar=False, values_format="d")
ax.set_title("Hand-written reviews (n=90) – tuned model, train-only"); ax.grid(False)
plt.tight_layout(); savefig("10_external_confusion.png"); plt.show()

# how much of the new text does the model's vocabulary even cover?
vocab = {w for w in names if " " not in w}
tokens = [t for txt in ext["review_text"] for t in clean_text(txt).split()]
oov = np.mean([t not in vocab for t in tokens])
print(f"Words in the hand-written reviews that the model has never seen: {oov:.0%}")

ext_acc = accuracy_score(ext["sentiment"], ext_pred)
METRICS["external_train_only"] = {"accuracy": ext_acc, "oov_rate": float(oov), "confusion_matrix": cm.tolist(),
                                  "f1_macro": precision_recall_fscore_support(ext["sentiment"], ext_pred, average="macro")[2]}

In [ ]:
wrong = ext.assign(pred=ext_pred)[ext_pred != ext["sentiment"]]
display(wrong.rename(columns={"sentiment": "true"}).head(12))

In [ ]:
import joblib, sklearn
from sklearn.base import clone

deployed = clone(final_search.best_estimator_)          # same vectoriser type + hyper-parameters
deployed.fit(df["review_text"], df["sentiment"])

MODEL_DIR = ROOT / "models"; MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(deployed, MODEL_DIR / "sentiment_pipeline.pkl", compress=3)

# reload and verify it behaves identically
reloaded = joblib.load(MODEL_DIR / "sentiment_pipeline.pkl")
assert (reloaded.predict(ext["review_text"]) == deployed.predict(ext["review_text"])).all()
size_kb = (MODEL_DIR / "sentiment_pipeline.pkl").stat().st_size / 1024
print(f"Saved models/sentiment_pipeline.pkl  ({size_kb:.0f} KB)")
print("Pipeline steps:", [n for n, _ in reloaded.steps], "| classes:", list(reloaded.classes_))

dep_ext_acc = accuracy_score(ext["sentiment"], reloaded.predict(ext["review_text"]))
print(f"Deployed model on the 90 hand-written reviews: accuracy = {dep_ext_acc:.1%}")
METRICS["external_deployed_accuracy"] = dep_ext_acc

In [ ]:
metadata = {
    "project": "AI-Powered Sentiment Analysis System",
    "trained_on": datetime.date.today().isoformat(),
    "sklearn_version": sklearn.__version__,
    "model": final_name,
    "features": "TF-IDF" if isinstance(deployed.named_steps["vec"], TfidfVectorizer) else "Bag of Words",
    "ngram_range": list(deployed.named_steps["vec"].ngram_range),
    "hyper_parameters": METRICS["final_params"],
    "vocabulary_size": len(deployed.named_steps["vec"].vocabulary_),
    "training_rows": int(len(df)),
    "classes": list(deployed.classes_),
    "metrics": {
        "cv_macro_f1_grouped": round(METRICS["final_cv_f1"], 4),
        "test_accuracy_unseen_sentences": round(final_eval["accuracy"], 4),
        "test_macro_f1_unseen_sentences": round(final_eval["f1_macro"], 4),
        "test_macro_roc_auc": round(final_eval["roc_auc_macro"], 4),
        "handwritten_accuracy_deployed": round(dep_ext_acc, 4),
    },
}
(MODEL_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

In [ ]:
from predict import predict_sentiment

tests = ["Absolutely love it, the quality is excellent and it arrived early!",
         "The battery died after two days and support never answered. Very disappointed.",
         "The package contains a charger and a user manual.",
         "It is not good at all."]
for t in tests:
    r = predict_sentiment(t, reloaded)
    probs = "  ".join(f"{c[:3]}={p:.0%}" for c, p in r["probabilities"].items())
    print(f"{r['label']:<9} {r['confidence']:.0%}  [{probs}]  key terms: {[w for w, _ in r['key_terms']]}\n   «{t}»")

In [ ]:
%%writefile app.py
"""
Streamlit web app for the AI-Powered Sentiment Analysis System.

Run with:   streamlit run app.py
The app only LOADS the saved pipeline (models/sentiment_pipeline.pkl);
it never re-trains anything.
"""
import pandas as pd
import streamlit as st

from predict import load_metadata, load_pipeline, predict_sentiment

st.set_page_config(page_title="Review Sentiment Analyzer", page_icon="💬", layout="centered")

STYLE = {
    "Positive": {"emoji": "😊", "color": "#2a9d8f"},
    "Neutral": {"emoji": "😐", "color": "#6c757d"},
    "Negative": {"emoji": "😞", "color": "#d1495b"},
}
EXAMPLES = {
    "— choose an example —": "",
    "Positive: great quality": "Excellent quality for the price, and I would happily buy it again.",
    "Negative: battery + support": "The battery drains quickly and customer support was slow and did not solve my issue.",
    "Neutral: factual": "The product comes with a charging adapter and a short cable.",
    "Tricky: negation": "The product is not good at all.",
}
LOW_CONFIDENCE = 0.60


@st.cache_resource(show_spinner="Loading model…")
def get_model():
    return load_pipeline()


try:
    pipeline = get_model()
except FileNotFoundError as err:
    st.error(str(err))
    st.stop()
meta = load_metadata()

# ------------------------------------------------------------------ sidebar
with st.sidebar:
    st.header("About the model")
    if meta:
        st.write(f"**{meta['model']}** on **{meta['features']}** "
                 f"(n-grams {tuple(meta['ngram_range'])})")
        m = meta["metrics"]
        st.metric("Accuracy on unseen sentences", f"{m['test_accuracy_unseen_sentences']:.1%}")
        st.metric("Macro F1 (test)", f"{m['test_macro_f1_unseen_sentences']:.3f}")
        st.metric("Accuracy on 90 hand-written reviews", f"{m['handwritten_accuracy_deployed']:.1%}")
        st.caption(f"Vocabulary: {meta['vocabulary_size']} terms · trained on {meta['training_rows']:,} reviews")
    st.info("The training data contains only ~70 distinct sentences, so unusual wording "
            "may be classified less reliably. Check the confidence score.")

# --------------------------------------------------------------------- main
st.title("💬 Review Sentiment Analyzer")
st.write("Type or paste a customer review and the model will classify it as "
         "**Positive**, **Negative** or **Neutral**.")

choice = st.selectbox("Try an example", list(EXAMPLES))
if choice != st.session_state.get("last_example"):
    st.session_state["last_example"] = choice
    if EXAMPLES[choice]:
        st.session_state["review_text"] = EXAMPLES[choice]

text = st.text_area("Your review", key="review_text", height=130,
                    placeholder="e.g. The delivery was late and the box was damaged…")
analyze = st.button("Analyze sentiment", type="primary")

if analyze:
    if not text.strip():
        st.warning("Please enter a review first.")
    else:
        result = predict_sentiment(text, pipeline)
        style = STYLE[result["label"]]

        st.markdown(
            f"""<div style="border-left:8px solid {style['color']};padding:14px 18px;
            border-radius:8px;background:{style['color']}18;margin:12px 0">
            <div style="font-size:0.9rem;opacity:.7">Predicted sentiment</div>
            <div style="font-size:2rem;font-weight:700;color:{style['color']}">
            {style['emoji']} {result['label']}</div>
            <div>Confidence: <b>{result['confidence']:.1%}</b></div></div>""",
            unsafe_allow_html=True)

        if result["confidence"] < LOW_CONFIDENCE:
            st.warning("Low confidence – the wording may be unfamiliar to the model, "
                       "so treat this prediction with care.")
        if not result["cleaned_text"]:
            st.warning("None of the words were recognised after preprocessing.")

        st.subheader("Probability for each class")
        for cls in ["Positive", "Neutral", "Negative"]:
            p = result["probabilities"][cls]
            st.write(f"{STYLE[cls]['emoji']} {cls} — {p:.1%}")
            st.progress(min(max(p, 0.0), 1.0))

        with st.expander("Why this prediction? (supporting information)"):
            if result["key_terms"]:
                terms = pd.DataFrame(result["key_terms"], columns=["word / phrase", "push towards " + result["label"]])
                st.dataframe(terms.round(3), hide_index=True, use_container_width=True)
            else:
                st.write("No strong key terms found.")
            st.caption("Text after preprocessing (what the model actually sees):")
            st.code(result["cleaned_text"] or "(empty)")

In [ ]:
%%writefile requirements.txt
pandas>=2.0
numpy>=1.26
scipy>=1.11
scikit-learn==1.9.1     # the pickled model was created with this version
joblib>=1.3
matplotlib>=3.8
seaborn>=0.13
streamlit>=1.35
jupyter>=1.0            # only needed to open the notebook

In [ ]:
!streamlit run app.py &>/content/streamlit_logs.txt &
!npm install -q localtunnel
!curl -s https://ipv4.icanhazip.com
!npx localtunnel --port 8501